# D1-S2-Nb4a: Fill the sample plates for Nb4
Transfers 2 x 50 uL of agonist into every well of each device's first sample plate: the 100 uL per well that Nb4's serial dilution starts from.

## Set your `protocol_mode`

...so it is easy to switch between simulation and execution

In [1]:
prep_simulation = False  # True or False

star_simulation = True  # True or False

flex_simulation = False  # True or False

num_samples = 96  # how many raw samples come in: 1 to 192 (2 plates)

run_identifier = "nb4a"

## Library Import

In [2]:
import asyncio
import logging
import time

---
## Visualizer3D
...so everything that follows has one highest-level reference frame

In [ ]:
from pylabrobot.resources import Coordinate, Resource
from pylabrobot.visualizer3D.facility import Facility
from pylabrobot.visualizer3D.server import Viewer3D

facility = Facility(name="facility", size_x=2_400, size_y=1_000, size_z=0)

viewer = Viewer3D(root = facility, name="nb4_normalisation")

await viewer.start()

viewer on http://127.0.0.1:1339/#token=FWE-ONUDeYdzX3gGw7dRlSAYD5fLhvxyvfwYFg1Cm2I  (websocket 2123)


viewer: a browser connected, drawing with WebGPU


---

## Set your `logging`
We (and our agents) always benefit from a log.

This follows standard Python logging but for convenience we set up:
- showing logging inside notebook using `verbose` declaration
- simple way to log to a file (in append mode)

I recommend always logging all commands you send/receive from any device for auditing purposes.

In [4]:
import datetime
import logging
import time

import pylabrobot
from pylabrobot.io import LOG_LEVEL_IO

# one mode per device: 8 combinations, each with its own log folder
prep_mode = "simulation" if prep_simulation else "execution"
star_mode = "simulation" if star_simulation else "execution"
flex_mode = "simulation" if flex_simulation else "execution"
protocol_mode = f"prep_{prep_mode}-star_{star_mode}-flex_{flex_mode}"

started_at = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
log_path = f"_logs/{protocol_mode}/{run_identifier}_{started_at}.log"
pylabrobot.setup_logger(log_path, level=LOG_LEVEL_IO)

pylabrobot.verbose(True, level=logging.INFO)

print(f"logging to {log_path}")
logging.getLogger("pylabrobot").info("--- %s (%s) ---", run_identifier, protocol_mode)

2026-09-28 13:30:11,304 - pylabrobot - INFO - --- nb4a (prep_execution-star_simulation-flex_execution) ---


logging to _logs/prep_execution-star_simulation-flex_execution/nb4a_20260928-133011.log


## Device setups
### (shadow) STAR(let)

In [5]:
from pylabrobot.hamilton.star import STARlet

star = STARlet(simulation=star_simulation)

facility.assign_child_resource(
    resource=star,
    location=Coordinate(x=800.0, y=0.0, z=0.0)
)

await star.setup()

2026-09-28 13:30:11,557 - pylabrobot.hamilton.star.driver.master - INFO - [Hamilton STAR] Connected on simulation (no link)
  Firmware: master 7.6S 25 2021_11_05 (GRU C0), pipettes 4.0S j 2022-03-16, x_arm 1.4S 2012-04-25, head96 5.0S i 2021-10-22 (H0 XE167), iswap 4.1S 2011-12-19, autoload 3.4S f 2017-01-09
  Arms: 1
  Configuration: 30 slots
  Autoload: 1D barcode scanner
    left: hamilton_legacy_star_dual_rail_arm, 354.0 mm wide, travel 95.0 to 800.2 mm, workspace -323.2 to 977.2 mm
      channels: 8 (1000uL) | 96-head: 96 head II | 384-head: none | iSWAP: wide gripper


### Opentrons Flex
The Flex has no simulator: its deck is built either way, and the robot is only reached when `flex_simulation = False`.
Then `setup()` creates a run and **homes all axes**.

In [6]:
from pylabrobot.opentrons import Flex
from pylabrobot.resources.opentrons import FlexDeck

flex_deck = FlexDeck()

facility.assign_child_resource(
    resource=flex_deck,
    location=Coordinate(x=-900.0, y=0.0, z=0.0)
)

flex = None
if not flex_simulation:
    flex = Flex(serial_port="/dev/tty.usbmodem1201", deck=flex_deck)  # Flex connected over USB serial
    # await flex.setup()

### Prep

In [7]:
from pylabrobot.hamilton.prep import Prep
from pylabrobot.hamilton.prep.driver.simulator import RECORDING_PREP_HEATER_SHAKER


HOST = "192.168.100.102"  # as in the Prep demo notebook; ignored in simulation
PORT = 2000

prep = Prep(
    simulation=prep_simulation,
    declared_configuration_json=RECORDING_PREP_HEATER_SHAKER,  # our Prep: 8-channel head, heater shaker
    host=HOST,
    port=PORT,
    )

facility.assign_child_resource(
    prep,
    location=Coordinate(x=0.0, y=0.0, z=0.0)
)



In [8]:
devices = [prep]
if not flex_simulation:
    devices.append(flex)

Parallelised setup of multiple devices... because why not :)

In [ ]:
# the Prep and the Flex run at the same time, each on its own
await asyncio.gather(
    *(
        device.setup() for device in devices
    )
)
await prep.lights.turn_on()

2026-09-28 13:30:12,671 - pylabrobot.hamilton.transport.tcp.tcp - INFO - Initializing Hamilton connection...
2026-09-28 13:30:12,674 - pylabrobot.io.http_serial - INFO - Opened HTTP-over-serial connection on /dev/tty.usbmodem1201
2026-09-28 13:30:12,836 - pylabrobot.hamilton.transport.tcp.tcp - INFO - Registering Hamilton client...
2026-09-28 13:30:12,838 - pylabrobot.hamilton.transport.tcp.tcp - INFO - Discovering Hamilton root objects...
2026-09-28 13:30:12,839 - pylabrobot.hamilton.transport.tcp.tcp - INFO - Discovering Hamilton global objects...
2026-09-28 13:30:12,843 - pylabrobot.hamilton.transport.tcp.tcp - INFO - Hamilton TCP client setup complete. Client ID: 7, globals: 1
2026-09-28 13:30:12,908 - pylabrobot.hamilton.transport.tcp.tcp - INFO - Initializing Hamilton connection...
2026-09-28 13:30:13,407 - pylabrobot.hamilton.transport.tcp.tcp - INFO - Registering Hamilton client...
2026-09-28 13:30:13,410 - pylabrobot.hamilton.transport.tcp.tcp - INFO - Discovering Hamilton roo

2026-09-28 13:40:13,879 - pylabrobot.hamilton.transport.tcp.session - WARNING - Dropping unmatched response from 1:1:48900 to 2:7:65535 seq=0 action=0x7
2026-09-28 13:40:13,881 - pylabrobot.hamilton.transport.tcp.session - WARNING - Dropping unmatched response from 1:1:48900 to 2:8:65535 seq=0 action=0x7
2026-09-28 13:40:13,881 - pylabrobot.hamilton.transport.tcp.session - WARNING - Hamilton TCP reader stopped: Connection closed before num_bytes are read


---
## Deck setup

we are now starting to build more complex setups and need more resources:

In [10]:
from pylabrobot.resources import set_tip_tracking, set_volume_tracking

set_tip_tracking(enabled=True)
set_volume_tracking(enabled=True)

In [11]:
import math

from pylabrobot.resources import ContainerRack, ResourceHolder, create_ordered_items_2d
from pylabrobot.resources.biorad import biorad_96_wellplate_200uL_Vb
from pylabrobot.resources.corning import cor_axy_1_troughplate_rowVb
from pylabrobot.resources.hamilton import (
    Trough_CAR_5R60_A00,
    hamilton_1_trough_60mL_Vb,
    hamilton_96_tiprack_50uL_NTR,
    hamilton_96_tiprack_300uL_NTR,
    hamilton_plate_carrier_L5_ac,
    hamilton_tip_carrier_L5_ntr_a00,
)

In [12]:
def hamilton_5_troughrack_60mL(name: str) -> ContainerRack:
    """5x 60 mL troughs, evenly spaced and centred on an SBS footprint."""
    trough_x, trough_y = 19.0, 90.0

    return ContainerRack(
        name=name,
        size_x=127.76,
        size_y=85.48,
        size_z=40.0,
        ordered_items=create_ordered_items_2d(
            klass=ResourceHolder,
            num_items_x=5,
            num_items_y=1,
            dx=10.0,
            dy=-2.3,  # the 90 mm trough overhangs the 85.48 mm footprint evenly
            dz=5.0,  # TODO: measure where the trough's base sits in the rack
            item_dx=21.5,
            item_dy=None,
            size_x=trough_x,
            size_y=trough_y,
            size_z=0.0,
            name_prefix=name,
        ),
    )

### `agonist`: one name on every device
Same name everywhere, prefixed with the device: `prep_agonist`, `star_agonist`, `flex_agonist`.

In [13]:
# Prep: the 60 mL trough in position 0 of the trough rack
prep_troughrack = prep.deck[0] = hamilton_5_troughrack_60mL(name="prep_troughrack")
prep_troughrack[0] = prep_agonist = hamilton_1_trough_60mL_Vb(name="prep_agonist")

# STAR: a 60 mL trough carrier, the 60 mL trough in position 0
star_trough_carrier = Trough_CAR_5R60_A00(name="star_trough_carrier")
star_trough_carrier[0] = star_agonist = hamilton_1_trough_60mL_Vb(name="star_agonist")
star.deck.assign_child_resource(star_trough_carrier, track=20)

# Flex: the 290 mL single-well reservoir
flex_agonist = cor_axy_1_troughplate_rowVb(name="flex_agonist")
flex_deck.assign_child_at_slot(flex_agonist, "D1")

# what is in each reservoir at the start, in uL
prep_agonist.tracker.set_volume(40_000)
star_agonist.tracker.set_volume(40_000)
flex_agonist.get_well("A1").tracker.set_volume(100_000)

### Sample plates
Each device has 2 dedicated positions for sample plates: in this simple example the plate the samples come in is also the plate they leave in.
How many plates are made follows from `num_samples`, declared at the top.

In [14]:
n_plates = math.ceil(num_samples / 96)
assert 1 <= n_plates <= 2, f"{num_samples} samples need {n_plates} plates; each device has 2 plate positions"


def new_plates(device: str) -> list:
    """Sample plates as they arrive: empty."""
    plates = [biorad_96_wellplate_200uL_Vb(name=f"{device}_sample_plate_{i}") for i in range(n_plates)]
    for plate in plates:
        plate.set_well_volumes([0.0] * plate.num_items)
    return plates

In [15]:
# Prep: deck sites 4, 5, right of the agonist's trough rack (site 0)
prep_plate_sites = [4, 5]

prep_sample_plates = new_plates("prep")
for site, plate in zip(prep_plate_sites, prep_sample_plates):
    prep.deck[site] = plate

In [16]:
# STAR: a plate carrier right of the agonist's trough carrier (track 20); positions 0, 1
star_plate_sites = [0, 1]

star_plate_carrier = hamilton_plate_carrier_L5_ac(name="star_plate_carrier")
star.deck.assign_child_resource(star_plate_carrier, track=21)

star_sample_plates = new_plates("star")
for site, plate in zip(star_plate_sites, star_sample_plates):
    star_plate_carrier[site] = plate

In [17]:
# Flex: slots D2, C2, right of the agonist (D1)
flex_plate_slots = ["D2", "C2"]

flex_sample_plates = new_plates("flex")
for slot, plate in zip(flex_plate_slots, flex_sample_plates):
    flex_deck.assign_child_at_slot(plate, slot)

then create new tipracks and assign them to the deck:

In [18]:
prep_tips_50uL_00 = prep.deck[1] = hamilton_96_tiprack_50uL_NTR(name="prep_tips_50uL_00")

prep_tips_300uL_00 = prep.deck[7] = hamilton_96_tiprack_300uL_NTR(name="prep_tips_300uL_00")

same on the STAR:

In [19]:
star_tip_carrier = hamilton_tip_carrier_L5_ntr_a00(name="star_tip_carrier")
star_tip_carrier[0] = star_tips_50uL_00 = hamilton_96_tiprack_50uL_NTR(name="star_tips_50uL_00")
star.deck.assign_child_resource(star_tip_carrier, track=13)

and on the Flex:

In [20]:
from pylabrobot.resources.opentrons import flex_96_tiprack_50ul

flex_tips_50uL_00 = flex_96_tiprack_50ul(name="flex_tips_50uL_00")
flex_deck.assign_child_at_slot(flex_tips_50uL_00, "C1")

---
## Fill: 2 x 50 uL of agonist into every column
Each multichannel (the Prep's 8-channel head, the STAR's 8 channels, the Flex's P50 8-channel on the right):
1. picks up a column of 50 uL tips,
2. for each column, 1 to 12, twice: aspirates 50 uL of agonist and dispenses it into the column,
3. returns the tips.

Every well then holds 100 uL: the starting point of Nb4.

In [21]:
# STAR: its 8 channels
channels = [0, 1, 2, 3, 4, 5, 6, 7]
tips = star_tips_50uL_00.column(0)
plate = star_sample_plates[0]
no_lld = star.pipettes.LLDMode.OFF

await star.pipettes.pick_up_tips(tip_spots=tips, use_channels=channels)

for col in range(12):
    for transfer in range(2):
        await star.pipettes.aspirate(
            containers=[star_agonist] * 8,
            volumes=[50] * 8,
            use_channels=channels,
            liquid_heights=[20] * 8,  # 40 mL stands ~32 mm; well above the floor
            lld_mode=no_lld,
            blow_out=[True] * 8,
        )
        await star.pipettes.dispense(
            containers=plate.column(col),
            volumes=[50] * 8,
            use_channels=channels,
            liquid_heights=[4] * 8,
            blow_out=[True] * 8,
        )

# the tips go back where they came from
await star.pipettes.return_tips(use_channels=channels)

In [22]:
from pylabrobot.lib.liquid_handling.tip_spot_finding import find_tip_spots

async def prep_fill():
    # Prep: its 8-channel head
    # the next full column: a column a previous run emptied is skipped
    tip_spots = find_tip_spots(prep_tips_50uL_00, has_tip=True, count=8, x_aligned=True)
    plate = prep_sample_plates[0]

    await prep.head8.pick_up_tips(tip_spots=tip_spots)

    for col in range(12):
        for transfer in range(2):
            await prep.head8.aspirate(
                containers=[prep_agonist],
                volume=50,
                lld_mode=prep.pipettes.LLDMode.CAPACITIVE,  # the head finds the agonist's surface
                blow_out=True,
            )
            await prep.head8.dispense(
                containers=plate.column(col), volume=50, liquid_height=4, blow_out=True
            )

    # the tips go back where they came from
    await prep.head8.drop_tips(destinations=tip_spots)

In [23]:
async def flex_fill():
    # Flex: the P50 8-channel on the right mount (only with a real Flex: it has no simulator)
    if flex is None:
        return
    p50 = flex.right
    plate = flex_sample_plates[0]

    await p50.pick_up_tips(target=flex_tips_50uL_00.column(0))

    for col in range(12):
        for transfer in range(2):
            await p50.aspirate_container(container=flex_agonist.get_well("A1"), volume=50)
            await p50.dispense(target=plate.column(col), volume=50, liquid_height=4)

    # the tips go back where they came from
    await p50.drop_tips(target=flex_tips_50uL_00, column=0)

In [24]:
# the Prep and the Flex run at the same time, each on its own
await asyncio.gather(prep_fill(), flex_fill())

2026-09-28 13:38:26,085 - pylabrobot.hamilton.prep.driver.features.head8 - INFO - [Prep MPH] pick_up_tips: rack=prep_tips_50uL_00, tip_spots=['A1', 'B1', 'C1', 'D1', 'E1', 'F1', 'G1', 'H1']
2026-09-28 13:38:26,506 - pylabrobot.opentrons.flex.flex - INFO - Loaded labware 'flex_tips_50uL_00' at slot C1 -> ID: dafc1bf7-f5e4-430f-8cb9-555578caf539
2026-09-28 13:38:32,509 - pylabrobot.hamilton.prep.driver.features.head8 - INFO - [Prep MPH] aspirate: resource=prep_troughrack_resourceholder_A1, containers=prep_agonist, volume=54.200, flow_rate=100.0
2026-09-28 13:38:45,725 - pylabrobot.hamilton.prep.driver.features.head8 - INFO - [Prep MPH] dispense: resource=prep_sample_plate_0, containers=['A1', 'B1', 'C1', 'D1', 'E1', 'F1', 'G1', 'H1'], volume=54.200, flow_rate=120.0
2026-09-28 13:38:53,578 - pylabrobot.hamilton.prep.driver.features.head8 - INFO - [Prep MPH] aspirate: resource=prep_troughrack_resourceholder_A1, containers=prep_agonist, volume=54.200, flow_rate=100.0
2026-09-28 13:39:06,713

[None, None]

---
# Shutdown

In [25]:
await star.stop()

In [26]:
await prep.stop()

2026-09-28 13:50:03,536 - pylabrobot.io.socket - INFO - Closing connection to socket 192.168.100.102:2000
2026-09-28 13:50:03,538 - pylabrobot.hamilton.transport.tcp.tcp - INFO - Hamilton TCP client stopped
2026-09-28 13:50:03,538 - pylabrobot.io.socket - INFO - Closing connection to socket 192.168.100.102:2000
2026-09-28 13:50:03,539 - pylabrobot.hamilton.transport.tcp.tcp - INFO - Hamilton TCP client stopped


In [27]:
if flex is not None:
    await flex.stop()  # homes and ends the run: it moves the robot

In [28]:
await viewer.stop()